In [59]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

metadata_dir = '/export/usuarios_ml4ds/danibacaicoa/ForwardBackard_losses_old/Datasets/raw_datasets/Clothing1M/'
clean_label_inst = os.path.join(metadata_dir, 'clean_label_kv.txt') # - Path, label for clean images 
noisy_label_inst = os.path.join(metadata_dir, 'noisy_label_kv.txt') # - Path, label for noisy images
clean_train_paths = os.path.join(metadata_dir, 'clean_train_key_list.txt') # - Path to the list of clean training images
noisy_train_paths = os.path.join(metadata_dir, 'noisy_train_key_list.txt') # - Path to the list of noisy training images
#clean_val_instances = os.path.join(metadata_dir, 'clean_val_key_list.txt')
clean_test_paths = os.path.join(metadata_dir, 'clean_test_key_list.txt') # - Path to the list of clean test images
category_names_eng = os.path.join(metadata_dir, 'category_names_eng.txt') # Cathegory names in English

def load_instances(filepath):
    '''Load labels from a file.
    Args:
        filepath (str): Path to the label file.
    Returns:
        dict: A dictionary mapping image paths to labels.
    '''
    labels = {}
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split()
            image_path = os.path.normpath(parts[0])
            labels[image_path] = int(parts[1])

    return labels

def load_paths(filepath):
    '''Load key list from a file.
    Args:
        filepath (str): Path to the key list file.
    Returns:
        list: A list of image paths.
    '''
    with open(filepath, 'r') as f:
        image_paths = [os.path.normpath(line.strip()) for line in f]
    return image_paths

def load_category_names(filepath):
    '''Load category names from a file.
    Args:
        filepath (str): Path to the category names file.
    Returns:
        list: A list of category names.
    '''
    with open(filepath, 'r') as f:
        category_names = [line.strip() for line in f]
    print(category_names)
    return category_names

class ClothingDataset(Dataset):
    def __init__(self, direction, train = "True", transform=None):
        self.direction = direction

        self.train = train
        self.transform = transform
        self.N = None

        self.c = len(load_category_names(category_names_eng))

        self.samples = []

        clean_instances = load_instances(clean_label_inst)
        noisy_instances = load_instances(noisy_label_inst)
        print(f"Loaded {len(clean_instances)} clean instances and {len(noisy_instances)} noisy instances.")

        num_clean = 0
        if self.train == "True":
            self.N = np.zeros((self.c, self.c))
            clean_train = load_paths(clean_train_paths)
            noisy_train = load_paths(noisy_train_paths)
            for key in noisy_train:
                if key in noisy_instances.keys():
                    self.samples.append((key, noisy_instances[key]))
                    if key in clean_instances.keys():
                        self.N[noisy_instances[key], clean_instances[key]] += 1
                        del clean_train[key]
            print(f"Loaded {len(noisy_train)} noisy training instances")
            for key in clean_train:
                self.samples.append((key, clean_instances[key]))
                num_clean += 1
            print(f"Loaded {num_clean} clean training instances")
            

        else:
            clean_test = load_paths(clean_test_paths)
            for key in clean_test:
                self.samples.append((key, clean_instances[key]))
            print(f"Loaded {len(clean_test)} clean test instances")


    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        relative_img_path, label = self.samples[idx]
        img_full_path = os.path.join(self.direction, relative_img_path)
        image = Image.open(img_full_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

img_size = 224
train_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

clothing1m_root = '/export/usuarios_ml4ds/danibacaicoa/ForwardBackard_losses_old/Datasets/raw_datasets/Clothing1M/' 

train_dataset = ClothingDataset(clothing1m_root, train = "True", transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
test_dataset = ClothingDataset(clothing1m_root, train = "False", transform=val_test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)   

['T-Shirt', 'Shirt', 'Knitwear', 'Chiffon', 'Sweater', 'Hoodie', 'Windbreaker', 'Jacket', 'Downcoat', 'Suit', 'Shawl', 'Dress', 'Vest', 'Underwear']
Loaded 72409 clean instances and 1037497 noisy instances.
Loaded 1000000 noisy training instances
Loaded 47570 clean training instances
['T-Shirt', 'Shirt', 'Knitwear', 'Chiffon', 'Sweater', 'Hoodie', 'Windbreaker', 'Jacket', 'Downcoat', 'Suit', 'Shawl', 'Dress', 'Vest', 'Underwear']
Loaded 72409 clean instances and 1037497 noisy instances.
Loaded 10526 clean test instances


In [60]:
clean_instances = load_instances(clean_label_inst)
noisy_instances = load_instances(noisy_label_inst)
clean_train = load_paths(clean_train_paths)
noisy_train = load_paths(noisy_train_paths)

In [63]:
for key in noisy_train:
    if key in clean_instances.keys():
        print(key)
        


In [48]:
len(pat)

1000000

In [49]:
pat[99999]

'images/1/10/2022270275,1524473110.jpg'

In [50]:
for i in pat:
    if i in c_inst:
        print(i)
        print(c_inst[i])
        print(n_inst[i])
        break

In [51]:
c_inst.values()

dict_values([9, 4, 8, 9, 9, 1, 4, 6, 11, 6, 6, 8, 12, 0, 0, 1, 8, 3, 1, 10, 11, 12, 10, 0, 8, 9, 3, 8, 10, 11, 5, 0, 6, 0, 8, 0, 8, 4, 9, 3, 0, 13, 13, 3, 8, 10, 4, 13, 5, 13, 5, 13, 9, 12, 8, 10, 6, 6, 4, 0, 8, 13, 13, 1, 8, 3, 8, 5, 12, 4, 4, 13, 3, 8, 7, 12, 10, 4, 3, 5, 2, 0, 3, 13, 12, 0, 3, 8, 8, 4, 10, 10, 1, 0, 0, 12, 0, 0, 7, 11, 0, 5, 8, 6, 8, 4, 8, 0, 2, 3, 5, 1, 8, 13, 10, 4, 6, 6, 11, 10, 5, 3, 0, 7, 11, 5, 2, 11, 2, 0, 0, 4, 5, 5, 5, 2, 9, 12, 2, 4, 13, 13, 2, 13, 13, 11, 10, 3, 5, 4, 11, 11, 4, 8, 6, 8, 5, 1, 5, 13, 7, 3, 11, 2, 4, 8, 13, 13, 5, 4, 5, 3, 4, 4, 11, 11, 13, 2, 5, 5, 3, 6, 3, 9, 10, 12, 9, 11, 13, 3, 6, 3, 8, 5, 13, 2, 11, 10, 11, 12, 13, 8, 5, 3, 8, 8, 12, 11, 3, 7, 8, 9, 4, 3, 5, 4, 10, 4, 7, 8, 4, 0, 7, 1, 3, 8, 8, 12, 10, 4, 4, 13, 8, 12, 5, 2, 11, 11, 11, 11, 13, 13, 8, 8, 6, 7, 13, 3, 8, 8, 3, 12, 4, 1, 4, 11, 8, 8, 6, 13, 4, 13, 10, 3, 12, 4, 3, 4, 8, 4, 7, 3, 10, 12, 4, 12, 8, 5, 0, 13, 4, 3, 9, 12, 2, 1, 2, 5, 5, 3, 8, 5, 3, 8, 8, 1, 5, 12, 4, 4, 3

In [58]:
pat[999999] in c_inst.keys()

False

In [ ]:
c_inst

In [ ]:
n_inst

In [64]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

metadata_dir = '/export/usuarios_ml4ds/danibacaicoa/ForwardBackard_losses_old/Datasets/raw_datasets/Clothing1M/'
clean_label_inst = os.path.join(metadata_dir, 'clean_label_kv.txt') # - Path, label for clean images 
noisy_label_inst = os.path.join(metadata_dir, 'noisy_label_kv.txt') # - Path, label for noisy images
clean_train_paths = os.path.join(metadata_dir, 'clean_train_key_list.txt') # - Path to the list of clean training images
noisy_train_paths = os.path.join(metadata_dir, 'noisy_train_key_list.txt') # - Path to the list of noisy training images
#clean_val_instances = os.path.join(metadata_dir, 'clean_val_key_list.txt')
clean_test_paths = os.path.join(metadata_dir, 'clean_test_key_list.txt') # - Path to the list of clean test images
category_names_eng = os.path.join(metadata_dir, 'category_names_eng.txt') # Cathegory names in English

def load_instances(filepath):
    '''Load labels from a file.
    Args:
        filepath (str): Path to the label file.
    Returns:
        dict: A dictionary mapping image paths to labels.
    '''
    labels = {}
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split()
            image_path = os.path.normpath(parts[0])
            labels[image_path] = int(parts[1])

    return labels

def load_paths(filepath):
    '''Load key list from a file.
    Args:
        filepath (str): Path to the key list file.
    Returns:
        list: A list of image paths.
    '''
    with open(filepath, 'r') as f:
        image_paths = [os.path.normpath(line.strip()) for line in f]
    return image_paths

def load_category_names(filepath):
    '''Load category names from a file.
    Args:
        filepath (str): Path to the category names file.
    Returns:
        list: A list of category names.
    '''
    with open(filepath, 'r') as f:
        category_names = [line.strip() for line in f]
    print(category_names)
    return category_names

class ClothingDataset(Dataset):
    def __init__(self, direction, train = "True", transform=None):
        self.direction = direction

        self.train = train
        self.transform = transform
        self.N = None

        self.c = len(load_category_names(category_names_eng))

        self.samples = []

        clean_instances = load_instances(clean_label_inst)
        noisy_instances = load_instances(noisy_label_inst)
        print(f"Loaded {len(clean_instances)} clean instances and {len(noisy_instances)} noisy instances.")

        if self.train == "True":
            self.N = np.zeros((self.c, self.c))
            clean_train = load_paths(clean_train_paths)
            noisy_train = load_paths(noisy_train_paths)
            clean_test = load_paths(clean_test_paths)

            for key in noisy_instances.keys():
                if key not in clean_test:
                    self.samples.append((key, noisy_instances[key]))
                    if key in clean_train and key in clean_test:
                        self.N[noisy_instances[key], clean_instances[key]] += 1

            

        else:
            clean_test = load_paths(clean_test_paths)
            for key in clean_test:
                self.samples.append((key, clean_instances[key]))
            print(f"Loaded {len(clean_test)} clean test instances")


    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        relative_img_path, label = self.samples[idx]
        img_full_path = os.path.join(self.direction, relative_img_path)
        image = Image.open(img_full_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

img_size = 224
train_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

clothing1m_root = '/export/usuarios_ml4ds/danibacaicoa/ForwardBackard_losses_old/Datasets/raw_datasets/Clothing1M/' 

train_dataset = ClothingDataset(clothing1m_root, train = "True", transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
test_dataset = ClothingDataset(clothing1m_root, train = "False", transform=val_test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)   

['T-Shirt', 'Shirt', 'Knitwear', 'Chiffon', 'Sweater', 'Hoodie', 'Windbreaker', 'Jacket', 'Downcoat', 'Suit', 'Shawl', 'Dress', 'Vest', 'Underwear']
Loaded 72409 clean instances and 1037497 noisy instances.
['T-Shirt', 'Shirt', 'Knitwear', 'Chiffon', 'Sweater', 'Hoodie', 'Windbreaker', 'Jacket', 'Downcoat', 'Suit', 'Shawl', 'Dress', 'Vest', 'Underwear']
Loaded 72409 clean instances and 1037497 noisy instances.
Loaded 10526 clean test instances


In [65]:
train_dataset.N

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [66]:
train_dataset.samples[0]

('images/2/46/50748838,3981129246.jpg', 5)

In [68]:
import numpy as np
import os

# --- Simplified Helper Function (minimal error checking) ---
def load_labels_simple(filepath):
    """Loads image path -> label mapping, assuming file exists and is valid."""
    labels = {}
    with open(filepath, 'r') as f:
        for line in f:
            try:
                parts = line.strip().split()
                if len(parts) == 2:
                    # Basic assumption: paths are consistent enough without normpath
                    image_path = parts[0]
                    labels[image_path] = int(parts[1])
            except:
                 # In a truly minimal version, even this try/except could be removed
                 # if perfect file format is guaranteed.
                 pass # Silently skip badly formatted lines
    return labels

# --- Simplified Estimation Function (core logic) ---
def estimate_transition_matrix_simple(clean_labels_path, noisy_labels_path, num_classes):
    """Estimates T[i, j] = P(noisy=j | clean=i) with minimal checks."""

    clean_labels = load_labels_simple(clean_labels_path)
    noisy_labels = load_labels_simple(noisy_labels_path)

    # Find images present in both sets
    common_keys = set(clean_labels.keys()) & set(noisy_labels.keys())
    if not common_keys:
        print("Warning: No common keys found between label files!")
        # Return an identity matrix or zeros as a fallback? Or raise error?
        # For simplicity, we'll let it proceed which might lead to div by zero later if empty
        pass

    # Initialize count matrix
    count_matrix = np.zeros((num_classes, num_classes), dtype=float)

    # Populate counts
    for key in common_keys:
        clean_label = clean_labels[key]
        noisy_label = noisy_labels[key]
        # Assume labels are valid indices [0, num_classes-1]
        if 0 <= clean_label < num_classes and 0 <= noisy_label < num_classes:
             count_matrix[clean_label, noisy_label] += 1.0
        # else: silently ignore invalid labels in this simplified version

    # Normalize rows to get probabilities
    # Add a small epsilon to prevent division by zero for rows with no counts
    epsilon = 1e-8
    row_sums = count_matrix.sum(axis=1, keepdims=True)

    # Normalize
    transition_matrix = count_matrix 

    # Optional: Handle rows that had zero sum (clean classes not seen in common_keys)
    # They will be all zeros due to epsilon. Depending on use case, might want
    # to set diagonal to 1 for those rows, or leave as is.
    # zero_sum_rows = np.where(row_sums < epsilon)[0]
    # if len(zero_sum_rows) > 0:
    #     print(f"Warning: Clean classes {zero_sum_rows} had no samples. Their rows are zero vectors.")


    return transition_matrix

# --- Main Execution (Simplified) ---

# !! IMPORTANT: Set this to your actual path for Clothing1M metadata !!
metadata_dir = '/export/usuarios_ml4ds/danibacaicoa/ForwardBackard_losses_old/Datasets/raw_datasets/Clothing1M/' # <-- User provided path

clean_label_kv_path = os.path.join(metadata_dir, 'clean_label_kv.txt')
noisy_label_kv_path = os.path.join(metadata_dir, 'noisy_label_kv.txt')

# Assume standard Clothing1M number of classes
num_classes = 14

print(f"Estimating transition matrix ({num_classes} classes) - Simplified Version")
# print(f"Clean labels: {clean_label_kv_path}") # Optional: uncomment to show paths
# print(f"Noisy labels: {noisy_label_kv_path}") # Optional: uncomment to show paths

# Calculate the matrix
T_simplified = estimate_transition_matrix_simple(
    clean_label_kv_path,
    noisy_label_kv_path,
    num_classes
)

# Print the resulting matrix
print("\nEstimated Transition Matrix T [P(noisy | clean)]:")
np.set_printoptions(precision=3, suppress=True) # Pretty print numpy array
print(T_simplified)
print(f"\nMatrix shape: {T_simplified.shape}")


Estimating transition matrix (14 classes) - Simplified Version

Estimated Transition Matrix T [P(noisy | clean)]:
[[1798.   84.  199.   64.   42.  166.    1.   78.    3.    7.    1.  195.
    39.  127.]
 [ 231. 2205.   32.  130.    4.    8.   11.   88.    7.   38.    1.  138.
    11.    8.]
 [ 188.   31.  607.   10.  379.   33.   10.    7.    1.    5.   23.   96.
    25.   14.]
 [ 601.  823.   62. 1820.    5.   11.    6.    9.    0.    4.    1.  518.
    90.    1.]
 [ 296.   13. 1280.    9.  230.   54.    6.    3.    0.    2.    4.   89.
    26.   42.]
 [  75.   29.  108.    5.   18. 1980.   96.  488.   30.   28.    2.   14.
    28.   31.]
 [   3.   23.   27.    0.    2.   34. 1232.  180.  445.  103.   11.   83.
     6.    5.]
 [   6.   99.    0.    0.    0.   36.  303.  586.   47.    3.    1.   44.
    13.    0.]
 [  13.   28.    6.    2.    1.   52.  560.  410. 1667.   11.   16.   13.
   344.   16.]
 [  16.   24.   33.    5.    3.    6.  440.  146.    2. 2497.    7.  120.
    11.    